# 04 — Direct 1+1D vs 3D validation

Runs individual BubbleMaster 1+1D simulations for a small set of
(lambda_bar, gamma) points and compares the resulting GW spectrum
directly against the corresponding sledgehamr 3D output.

**Workflow**
1. Fill in `VALIDATION_POINTS` with your (lambda_bar, gamma, sledgehamr_path) triples.
2. Run **Section 1** to write setup HDF5 files and SLURM scripts.
3. Submit jobs: `! sbatch scripts/validate_<tag>.sh`
4. After jobs complete, run **Section 2** onward to load and compare.

In [9]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
import ptbuilder as pt
from pathlib import Path
from ptbuilder.ic import _two_bubble_ic, _cutoff_times, BubbleMasterParams

# Bubble locations in sledgehamr reference
xlocs = [7.225e+1, 1.278e+2, 1.087e+2]
ylocs = [8.992e+1, 8.992e+1, 1.101e+2]
zlocs = [1.000e+2, 1.000e+2, 1.000e+2]
pos = np.column_stack((xlocs, ylocs, zlocs))
distances = np.linalg.norm(pos[:, None, :] - pos[None, :, :], axis=-1)
distances = np.unique(distances)
print(distances)

# slightly different rounding for the special gamma_2 run
distance_gamma2 = 1.139e+2 - 8.615e+1

[ 0.         27.78565097 41.663352   55.55      ]


In [10]:
# Path to pySledgehamr — adjust to your cluster location
PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)

# Box parameters for sledgehamr runs (must match what was simulated)
SH_L        = 200.    # box side length
SH_ZERO_PAD = 2.      # zero-padding factor used in spectrum output

# -----------------------------------------------------------------------
# Validation points: specify lambda_bar and d (bubble separation as used
# in the sledgehamr run).  gamma is computed from d via the wall-thinning
# convention so the BubbleMaster setup uses exactly the same separation.
# Fill in real d values once you look them up.
# -----------------------------------------------------------------------
sh_output_base = '/pscratch/sd/b/buschman/other_runs/' 
VALIDATION_POINTS = [
    dict(lambda_bar=0.84, d=distance_gamma2, sh_path=sh_output_base+'bubble_normalization_gamma2'),
    dict(lambda_bar=0.84, d=distances[2], sh_path=sh_output_base+'bubble_N3_234_0_2_'),
    dict(lambda_bar=0.84, d=distances[3], sh_path=sh_output_base+'bubble_N3_234_0_1_'),
]

config = pt.Config()
params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)


def gamma_from_d(model, d):
    """
    Compute the wall-thinning Lorentz factor corresponding to bubble
    separation d.  Inverse of _two_bubble_ic: given d, returns gamma
    such that _two_bubble_ic(profile, gamma) reproduces exactly d.
    """
    kin   = model.kinematics
    t_c   = kin.collision_time(d)
    return float(kin.Gamma(t_c))

## 1. Write setup files and SLURM scripts

In [11]:
scripts_dir = Path('scripts')
scripts_dir.mkdir(exist_ok=True)

for vp in VALIDATION_POINTS:
    lb = vp['lambda_bar']
    d  = vp['d']

    model   = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    gamma   = gamma_from_d(model, d)
    vp['gamma'] = gamma   # store for later use

    tag = f'lb{lb}_g{gamma:.4g}'

    # Time grid derived from physical scales
    t_0, t_cut, t_m, t_max = _cutoff_times(d, params.cutoff_type, params.t_0_scal)
    times = np.linspace(0.5 * t_cut, 1.1 * t_max, 20)

    run_dir    = model.results_dir / 'validation' / tag
    setup_path = run_dir / 'setup.h5'
    out_dir    = run_dir / 'out'
    run_dir.mkdir(parents=True, exist_ok=True)

    pt.write_2d_setup(model, gamma=gamma, times=times,
                      path=setup_path, params=params)

    print(f'[{tag}]')
    print(f'  d={d:.4f}  gamma(wall-thinning)={gamma:.6f}')
    print(f'  t_cut={t_cut:.3f}  t_max={t_max:.3f}')
    print(f'  setup → {setup_path}')

    # --- SLURM script ---
    script_path = scripts_dir / f'validate_{tag}.sh'
    slurm = f"""#!/bin/bash
#SBATCH --constraint=cpu
#SBATCH --nodes=1
#SBATCH --tasks-per-node=1
#SBATCH --cpus-per-task=128
#SBATCH --qos=regular
#SBATCH --time=02:00:00
#SBATCH --job-name=bm_{tag}
#SBATCH --output=logs/bm_{tag}_%j.out

cd $SLURM_SUBMIT_DIR
export SLURM_CPU_BIND=cores
export OMP_PLACES=threads
export OMP_PROC_BIND=spread
export OMP_NUM_THREADS=128

mkdir -p {out_dir}
srun bin/bubblemaster {setup_path} {out_dir}/
"""
    script_path.write_text(slurm)
    print(f'  script → {script_path}')
    print()

Path('logs').mkdir(exist_ok=True)
print('Done. Submit with:  sbatch scripts/validate_<tag>.sh')

[lb0.84_g1.937]
  d=27.7500  gamma(wall-thinning)=1.937051
  t_cut=29.970  t_max=35.797
  setup → data/phi4_lambda_bar0.84/validation/lb0.84_g1.937/setup.h5
  script → scripts/validate_lb0.84_g1.937.sh

[lb0.84_g2.962]
  d=41.6634  gamma(wall-thinning)=2.962170
  t_cut=44.996  t_max=53.746
  setup → data/phi4_lambda_bar0.84/validation/lb0.84_g2.962/setup.h5
  script → scripts/validate_lb0.84_g2.962.sh

[lb0.84_g3.974]
  d=55.5500  gamma(wall-thinning)=3.973938
  t_cut=59.994  t_max=71.659
  setup → data/phi4_lambda_bar0.84/validation/lb0.84_g3.974/setup.h5
  script → scripts/validate_lb0.84_g3.974.sh

Done. Submit with:  sbatch scripts/validate_<tag>.sh


Submit jobs from the repo root (not from notebooks/):
```bash
cd /path/to/PhaseTransitionBuilder
sbatch scripts/validate_lb0.84_g2.0.sh
sbatch scripts/validate_lb0.84_g3.0.sh
sbatch scripts/validate_lb0.84_g4.0.sh
```
---
## 2. Load BubbleMaster results (run after jobs complete)

In [ ]:
def load_bm_result(out_dir):
    """Load the final cumulative BubbleMaster spectrum from result_*.h5 files."""
    out_dir = Path(out_dir)
    files = sorted(out_dir.glob('result_*.h5'),
                   key=lambda p: int(p.stem.split('_')[1]))
    if not files:
        raise FileNotFoundError(f'No result_*.h5 in {out_dir}')
    with h5py.File(files[-1], 'r') as f:
        w    = f['w'][:]
        spec = f['spectrum'][:]
    return w, spec


bm_results = {}
for vp in VALIDATION_POINTS:
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma}'

    model   = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    out_dir = model.results_dir / 'validation' / tag / 'out'

    with h5py.File(model.results_dir / 'validation' / tag / 'setup.h5', 'r') as f:
        d = float(f.attrs['d'])

    try:
        w, spec = load_bm_result(out_dir)
        bm_results[tag] = dict(w=w, spec=spec, d=d, lb=lb, gamma=gamma)
        print(f'[{tag}]  loaded {len(w)} frequencies,  d={d:.3f}')
    except FileNotFoundError as e:
        print(f'[{tag}]  MISSING: {e}')

## 3. Load sledgehamr 3D spectra

In [ ]:
def load_sledgehamr_spectrum(sh_path, snapshot_index=-1, L=200., zero_pad=2.):
    """
    Load GW spectrum from a sledgehamr output directory.
    Tries pySledgehamr first; falls back to reading spectra.hdf5 directly.

    Returns (k_sq, spectrum, t) where k_sq are integer shell indices.
    """
    try:
        import pySledgehamr as sledgehamr
        sh_out = sledgehamr.Output(sh_path)
        n_snaps = len(sh_out.gravitational_wave_spectra)
        idx = snapshot_index if snapshot_index >= 0 else n_snaps + snapshot_index
        data = sh_out.GetGravitationalWaveSpectrum(idx)
        return data['k_sq'], data['spectrum'], float(data['t'])
    except (ImportError, Exception):
        pass

    # Fallback: read spectra.hdf5 directly
    spectra_file = Path(sh_path) / 'spectra.hdf5'
    if not spectra_file.exists():
        # Try common sledgehamr output patterns
        for candidate in Path(sh_path).rglob('spectra.hdf5'):
            spectra_file = candidate
            break
    if not spectra_file.exists():
        raise FileNotFoundError(f'spectra.hdf5 not found in {sh_path}')

    with h5py.File(spectra_file, 'r') as f:
        keys = sorted(f.keys(), key=lambda k: int(k) if k.isdigit() else 0)
        key  = keys[snapshot_index]
        grp  = f[key]
        k_sq     = grp['k_sq'][:]
        spectrum = grp['spectrum'][:]
        t        = float(grp.attrs.get('t', grp.attrs.get('time', 0.)))
    return k_sq, spectrum, t


def normalize_sh_spectrum(k_sq, spectrum, d, L=200., zero_pad=2.):
    """Convert raw sledgehamr spectrum to dE/dlnk vs kR* for comparison."""
    Leff = L * zero_pad
    kIR  = 2 * np.pi / Leff
    r    = np.sqrt(k_sq)
    k    = r * kIR
    kR   = k * d                          # x-axis: kR*
    kL   = 2 * np.pi * r
    norm = 4.0 * kL * (Leff ** 3)        # converts to dE/dlnk
    return kR, spectrum * norm


sh_results = {}
for vp in VALIDATION_POINTS:
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma}'

    try:
        k_sq, spec, t = load_sledgehamr_spectrum(
            vp['sh_path'], snapshot_index=-1, L=SH_L, zero_pad=SH_ZERO_PAD)
        d = bm_results[tag]['d'] if tag in bm_results else vp.get('d', None)
        sh_results[tag] = dict(k_sq=k_sq, spec=spec, t=t, d=d)
        print(f'[{tag}]  sledgehamr snapshot at t={t:.2f}')
    except Exception as e:
        print(f'[{tag}]  sledgehamr not loaded: {e}')

## 4. Compare 1+1D vs 3+1D

In [ ]:
colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']

fig, axes = plt.subplots(1, len(VALIDATION_POINTS),
                         figsize=(5 * len(VALIDATION_POINTS), 5),
                         sharey=True)
if len(VALIDATION_POINTS) == 1:
    axes = [axes]

for ax, vp, color in zip(axes, VALIDATION_POINTS, colors):
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma}'

    # 1+1D
    if tag in bm_results:
        r  = bm_results[tag]
        kR = r['w'] * r['d']     # kR* = k * R* = w * d
        ax.plot(kR, r['spec'], color=color, linestyle=':', lw=2, label='1+1D')

    # 3+1D
    if tag in sh_results:
        s = sh_results[tag]
        d = s['d'] or (bm_results[tag]['d'] if tag in bm_results else 1.)
        kR_sh, y_sh = normalize_sh_spectrum(
            s['k_sq'], s['spec'], d, L=SH_L, zero_pad=SH_ZERO_PAD)
        # mode-count error bands (1σ)
        r_sh   = np.sqrt(s['k_sq'])
        N_modes = np.maximum(1., 4 * np.pi * r_sh ** 2)
        yerr   = y_sh * np.sqrt(2. / N_modes)
        ax.plot(kR_sh, y_sh, color=color, linestyle='-', lw=2,
                label=f'3+1D  (t={s["t"]:.1f})')
        ax.fill_between(kR_sh,
                        np.maximum(y_sh - yerr, 1e-300),
                        y_sh + yerr,
                        color=color, alpha=0.2, linewidth=0)

    ax.set_title(rf'$\lambda_\mathrm{{bar}}={lb}$,  $\gamma={gamma}$')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$k R_*$')
    ax.legend(frameon=False, fontsize=9)

axes[0].set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
fig.suptitle('Direct 1+1D vs 3+1D comparison', y=1.01)
plt.tight_layout()
plt.savefig('validate_direct.pdf', bbox_inches='tight')
plt.show()